# Dashboards from the cleaned data

- the question - Do equity funds with a higher expense ratio return less, and how large is the gap between direct and regular plans?


In [87]:
# imports
try:
    import dash,dash_bootstrap_components
except ImportError:
    !pip install dash dash-bootstrap-components
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

# Dashboard1

In [88]:
nav = pd.read_parquet('data_collection/clean/nav_midcap_clean.parquet')
ter = pd.read_parquet('data_collection/clean/ter_midcap_clean.parquet')
DIRECT,REGULAR = "#38BDF8","#FB923C"
COUNT = "#A78BFA"
GRID = "#343B46"
CARDS = "#282828"
TEXT = "#E6EDF3"
MUTED = "#9AA7B4"
THEME = dbc.themes.CYBORG
print(ter.info())
print("-"*40) 
print(nav.info())  
print("-"*40)
print(ter["MF_ID"].nunique())
print(nav["MF_ID"].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 68236 entries, 0 to 68235
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Scheme_Name        68236 non-null  string        
 1   R_BaseTER          62315 non-null  float64       
 2   R_6A_B             62315 non-null  float64       
 3   R_6A_C             62315 non-null  float64       
 4   R_GST              62315 non-null  float64       
 5   R_TER              68172 non-null  float64       
 6   D_BaseTER          62315 non-null  float64       
 7   D_6A_B             62315 non-null  float64       
 8   D_6A_C             62315 non-null  float64       
 9   D_GST              62315 non-null  float64       
 10  D_TER              67341 non-null  float64       
 11  MF_ID              68236 non-null  int64         
 12  R_BER              5921 non-null   float64       
 13  R_BrokerageCost    5921 non-null   float64       
 14  R_TransactionCost

In [89]:
ter = ter[ter["MF_ID"].isin(nav["MF_ID"].unique())].copy()
companies = (nav.drop_duplicates(subset=["MF_ID"])[["MF_ID","Mutual_Fund_Company"]].assign(COMPANY=lambda x: x["Mutual_Fund_Company"].str.replace("Mutual Fund","",regex=False).str.strip()))
print(companies.info())
print(companies)

<class 'pandas.DataFrame'>
Index: 33 entries, 0 to 93746
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   MF_ID                33 non-null     int64
 1   Mutual_Fund_Company  33 non-null     str  
 2   COMPANY              33 non-null     str  
dtypes: int64(1), str(2)
memory usage: 2.0 KB
None
       MF_ID                Mutual_Fund_Company                COMPANY
0          3  Aditya Birla Sun Life Mutual Fund  Aditya Birla Sun Life
3822       4     Baroda BNP Paribas Mutual Fund     Baroda BNP Paribas
6066       6                    DSP Mutual Fund                    DSP
9888       9                   HDFC Mutual Fund                   HDFC
13710     13                  quant Mutual Fund                  quant
17534     16           JM Financial Mutual Fund           JM Financial
19432     17         Kotak Mahindra Mutual Fund         Kotak Mahindra
23254     18                    LIC Mutual Fund     

- KPI Cards

In [90]:
# ---------- numbers for the overview ----------
latest = (ter.dropna(subset=["R_TER", "D_TER"]).sort_values("Date").groupby("MF_ID").tail(1))
n_funds = nav["MF_ID"].nunique()
n_rows  = len(nav)
years   = (nav["Date"].max() - nav["Date"].min()).days / 365.25
avg_r, avg_d = latest["R_TER"].mean(), latest["D_TER"].mean()
avg_gap = (latest["R_TER"] - latest["D_TER"]).mean()

# one row per fund COMPANY: first NAV date, last NAV date
span = (nav.groupby("MF_ID")["Date"].agg(start="min", end="max").reset_index().merge(companies, on="MF_ID"))
span["years_of_history"] = (span["end"] - span["start"]).dt.days / 365.25
span["month"] = span["start"].dt.to_period("M").dt.to_timestamp()

# how many fund companies have a NAV in each month, and who joined that month
active = (nav.groupby(nav["Date"].dt.to_period("M"))["MF_ID"].nunique().rename("funds").reset_index())
active["Date"] = active["Date"].dt.to_timestamp()
joined = span.groupby("month")["COMPANY"].apply(lambda s: "<br>".join("• " + h for h in sorted(s)))
active["note"] = [("Joined:<br>" + joined[d]) if d in joined.index else "" for d in active["Date"]]

first_n, last_n = int(active["funds"].iloc[0]), int(active["funds"].iloc[-1])
active.loc[0, "note"] = f"{first_n} companies were already in the data"

newcomers = span[span["start"] > nav["Date"].min()]              # not the day-one group
busiest_year  = int(newcomers["start"].dt.year.value_counts().idxmax())
busiest_count = int(newcomers["start"].dt.year.value_counts().max())
newest = span.sort_values("start").iloc[-1]

COMPANY_options = [{"label": h, "value": str(i)}
                 for i, h in sorted(zip(companies["MF_ID"], companies["COMPANY"]), key=lambda x: x[1].lower())]
default_id = int(next((o["value"] for o in COMPANY_options if o["label"] == "HDFC"), COMPANY_options[0]["value"]))

In [ ]:
# ---------- helpers ----------
def style_fig(fig, height=380):
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="tint(0,0,0,0)",
        plot_bgcolor="tint(0,0,0,0)",

        font=dict(
            family="Inter, Segoe UI, sans-serif",
            color=TEXT,
            size=13
        ),

        height=height,

        margin=dict(l=50, r=35, t=25, b=45),

        hoverlabel=dict(
            bgcolor="#1B2430",
            bordercolor=GRID,
            font=dict(color=TEXT, size=14)
        ),

        legend=dict(
            orientation="h",
            y=1.12,
            x=0
        )
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        tickfont=dict(color=MUTED)
    )

    fig.update_yaxes(
        gridcolor=GRID,
        zeroline=False,
        tickfont=dict(color=MUTED)
    )

    return fig

def graph(fig, height=380, **kw):
    return dcc.Graph(figure=style_fig(fig, height), config={"displayModeBar": False}, **kw)

def kpi(label, value, color=None, sub=None):
    body = [html.P(label, className="text-muted mb-1"),
            html.H3(value, style={"color": color} if color else {})]
    if sub: body.append(html.Small(sub, className="text-muted"))
    return dbc.Card(dbc.CardBody(body), className="h-100")

def kpi_two(label, d_val, r_val, sub=None):
    """A card that shows the Direct and Regular value on two lines."""
    line = lambda name, val, col: html.Div([html.Span(name + "  ", className="text-muted"),
                                            html.Span(val, style={"color": col, "fontWeight": 600, "fontSize": "1.5rem"})])
    body = [html.P(label, className="text-muted mb-1"), line("Direct", d_val, DIRECT), line("Regular", r_val, REGULAR)]
    if sub: body.append(html.Small(sub, className="text-muted"))
    return dbc.Card(dbc.CardBody(body), className="h-100")

def chart_card(title, subtitle, content):
    return dbc.Card(dbc.CardBody([html.H5(title, className="mb-0"),
                                  html.Small(subtitle, className="text-muted"), content]), className="h-100")

fmt = lambda x, f: "n/a" if pd.isna(x) else format(x, f)

# ---------- chart 1: how many fund COMPANYs over time ----------
ends = active.iloc[[0, -1]]
fig_active = go.Figure()
fig_active.add_trace(go.Scatter(
    x=active["Date"], y=active["funds"], mode="lines",
    line=dict(color=COUNT, width=3, shape="hv"),
    fill="tozeroy", fillcolor="tint(167,139,250,0.14)",
    customdata=active["note"],
    hovertemplate="<b>%{y} fund companies</b> · %{x|%b %Y}<br>%{customdata}<extra></extra>"))
fig_active.add_trace(go.Scatter(                   # direct labels on the first and last point
    x=ends["Date"], y=ends["funds"], mode="markers+text", text=ends["funds"].astype(str),
    textposition="top center", textfont=dict(color=COUNT, size=18),
    marker=dict(color=COUNT, size=10), hoverinfo="skip", showlegend=False))
fig_active.update_yaxes(range=[0, last_n * 1.25], dtick=10)
fig_active.update_layout(showlegend=False)

# ---------- the TER chart: monthly average, lightly smoothed, labelled directly ----------
def make_ter_fig(t):
    fig = go.Figure()
    tm = (t.set_index("Date")[["D_TER", "R_TER"]].resample("MS").mean()
            .dropna(how="all").rolling(3, min_periods=1).mean())
    both = tm.dropna()
    if both.empty:                                   # a COMPANY with no TER data
        fig.add_annotation(text="No TER data for this COMPANY", showarrow=False,
                           font=dict(color="#8B98A5", size=14))
        return fig

    last = both.iloc[-1]
    last_x = both.index[-1]
    gap_now = last["R_TER"] - last["D_TER"]

    fig.add_trace(go.Scatter(x=tm.index, y=tm["D_TER"], name="Direct",
                             line=dict(color=DIRECT, width=3, shape="spline", smoothing=0.6),
                             hovertemplate="%{y:.2f}%"))
    fig.add_trace(go.Scatter(x=tm.index, y=tm["R_TER"], name="Regular",
                             line=dict(color=REGULAR, width=3, shape="spline", smoothing=0.6),
                             fill="tonexty", fillcolor="tint(251,146,60,0.16)",
                             hovertemplate="%{y:.2f}%"))
    # direct labels at the end of each line; cliponaxis=False lets them sit outside the plot area
    for name, col, val in [("Regular", REGULAR, last["R_TER"]), ("Direct", DIRECT, last["D_TER"])]:
        fig.add_trace(go.Scatter(x=[last_x], y=[val], mode="markers+text",
                                 text=[f"  {name} {val:.2f}%"], textposition="middle right",
                                 textfont=dict(color=col, size=13), marker=dict(color=col, size=9),
                                 cliponaxis=False, hoverinfo="skip", showlegend=False))
    fig.add_annotation(x=last_x, y=(last["R_TER"] + last["D_TER"]) / 2,
                       text=f"  gap {gap_now:.2f} pp", showarrow=False, xanchor="left",
                       font=dict(color="#8B98A5", size=12))

    fig.update_layout(hovermode="x unified", showlegend=False, margin=dict(r=150))
    fig.update_xaxes(range=[tm.index[0], last_x])    # stop the axis at the last data point (no empty 2028)
    fig.update_yaxes(ticksuffix="%", tickformat=".1f", rangemode="tozero")
    return fig

# ---------- the profile of one fund COMPANY ----------
def profile(mf_id):
    n = nav[nav["MF_ID"] == mf_id].dropna(subset=["NAV"]).sort_values("Date")
    t = ter[ter["MF_ID"] == mf_id].sort_values("Date")
    COMPANY = companies.loc[companies["MF_ID"] == mf_id, "COMPANY"].iloc[0]
    scheme = t["scheme_label"].iloc[-1] if ("scheme_label" in t and len(t)) else ""
    sp = span[span["MF_ID"] == mf_id].iloc[0]

    def nav_stats(plan):
        s = n.loc[n["Plan"] == plan, "NAV"]
        if s.empty: return dict(start=np.nan, now=np.nan, avg=np.nan, chg=np.nan)
        return dict(start=s.iloc[0], now=s.iloc[-1], avg=s.mean(), chg=(s.iloc[-1] / s.iloc[0] - 1) * 100)

    def ter_stats(col):
        s = t[col].dropna()
        if s.empty: return dict(start=np.nan, now=np.nan, avg=np.nan, lo=np.nan, hi=np.nan)
        return dict(start=s.iloc[0], now=s.iloc[-1], avg=s.mean(), lo=s.min(), hi=s.max())

    nd, nr = nav_stats("Direct Plan"), nav_stats("Regular Plan")
    td, tr_ = ter_stats("D_TER"), ter_stats("R_TER")

    title = html.Div([html.H3(COMPANY, className="mb-0"), html.Span(scheme, className="text-muted")])

    cards = dbc.Row([
        dbc.Col(kpi("In our data since", f"{sp['start']:%d %b %Y}", sub=f"{sp['years_of_history']:.1f} years of history"), width=3),
        dbc.Col(kpi_two("NAV now (₹ per unit)", fmt(nd["now"], ",.2f"), fmt(nr["now"], ",.2f"), f"as of {n['Date'].max():%d %b %Y}"), width=3),
        dbc.Col(kpi_two("TER now (% a year)", fmt(td["now"], ".2f") + "%", fmt(tr_["now"], ".2f") + "%",
                        f"gap {fmt(tr_['now'] - td['now'], '.2f')} pp"), width=3),
        dbc.Col(kpi_two("Average TER", fmt(td["avg"], ".2f") + "%", fmt(tr_["avg"], ".2f") + "%", "over its whole history"), width=3),
    ], className="mb-3")

    # NAV chart
    fig_nav = go.Figure()
    for plan, col in [("Direct Plan", DIRECT), ("Regular Plan", REGULAR)]:
        s = n[n["Plan"] == plan]
        fig_nav.add_trace(go.Scatter(x=s["Date"], y=s["NAV"], name=plan.replace(" Plan", ""),
                                     line=dict(color=col, width=2), hovertemplate="%{y:,.2f}"))
    fig_nav.update_layout(hovermode="x unified")
    fig_nav.update_yaxes(title_text="NAV (₹)")

    # TER chart (new)
    fig_ter = make_ter_fig(t)

    # numbers table (color="dark" removed so the theme styles it)
    rows = [("NAV at start (₹)",      fmt(nd["start"], ",.2f"), fmt(nr["start"], ",.2f")),
            ("NAV now (₹)",           fmt(nd["now"], ",.2f"),   fmt(nr["now"], ",.2f")),
            ("NAV change since start", fmt(nd["chg"], "+,.0f") + "%", fmt(nr["chg"], "+,.0f") + "%"),
            ("Average NAV (₹)",       fmt(nd["avg"], ",.2f"),   fmt(nr["avg"], ",.2f")),
            ("TER at start",          fmt(td["start"], ".2f") + "%", fmt(tr_["start"], ".2f") + "%"),
            ("TER now",               fmt(td["now"], ".2f") + "%",   fmt(tr_["now"], ".2f") + "%"),
            ("Average TER",           fmt(td["avg"], ".2f") + "%",   fmt(tr_["avg"], ".2f") + "%"),
            ("TER lowest to highest", f"{fmt(td['lo'], '.2f')}% to {fmt(td['hi'], '.2f')}%",
                                      f"{fmt(tr_['lo'], '.2f')}% to {fmt(tr_['hi'], '.2f')}%")]
    table = dbc.Table([
        html.Thead(html.Tr([html.Th(""), html.Th("Direct", style={"color": DIRECT}), html.Th("Regular", style={"color": REGULAR})])),
        html.Tbody([html.Tr([html.Td(a, className="text-muted"), html.Td(b), html.Td(c)]) for a, b, c in rows])],
        hover=True, className="mb-0")
    return title, cards, fig_nav, fig_ter, table

In [92]:
app = Dash(__name__, external_stylesheets=[THEME])

title0, cards0, fig_nav0, fig_ter0, table0 = profile(default_id)     # first view is drawn without a callback

app.layout = dbc.Container([
    # ---- header ----
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 1 · Who is in our data, and what does each fund COMPANY look like?",
                            className="text-muted")], width=12), className="mb-3"),

    # ---- overview KPIs (2+2+2+2+2+2) ----
    dbc.Row([
        dbc.Col(kpi("Fund COMPANYs", f"{n_funds}"), width=2),
        dbc.Col(kpi("NAV rows", f"{n_rows:,}"), width=2),
        dbc.Col(kpi("Years of data", f"{years:.1f}"), width=2),
        dbc.Col(kpi("Avg Regular TER", f"{avg_r:.2f}%", REGULAR), width=2),
        dbc.Col(kpi("Avg Direct TER", f"{avg_d:.2f}%", DIRECT), width=2),
        dbc.Col(kpi("Avg gap", f"{avg_gap:.2f} pp"), width=2),
    ], className="mb-4"),

    # ---- how many funds over time (8 + 4) ----
    dbc.Row([
        dbc.Col(chart_card("How many mid-cap fund COMPANYs are in our data?",
                           f"Grew from {first_n} to {last_n}. Hover a step to see who joined.",
                           graph(fig_active, 340)), width=8),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("What this tells us"),
            dcc.Markdown(f"""
- **{first_n}** COMPANYs were there on day one. Now there are **{last_n}** (**+{last_n - first_n}**).
- **{busiest_year}** brought the most new COMPANYs (**{busiest_count}**).
- The newest is **{newest['COMPANY']}**, first seen {newest['start']:%d %b %Y}.
- Younger funds have less history, so later dashboards compare funds on **common time windows**.
            """)]), className="h-100"), width=4),
    ], className="mb-4"),

    # ---- fund COMPANY explorer ----
    dbc.Row([
        dbc.Col([html.H4("Explore a fund COMPANY", className="mb-0"),
                 html.Small("Pick a COMPANY to see when it started, its NAV and TER, now and on average.", className="text-muted")], md=8),
        dbc.Col(dcc.Dropdown(id="COMPANY-select", options=COMPANY_options, value=str(default_id), clearable=False), md=4),
    ], className="mb-3 align-items-end"),

    dbc.Row(dbc.Col(html.Div(title0, id="COMPANY-title"), width=12), className="mb-2"),
    html.Div(cards0, id="COMPANY-cards"),

    dbc.Row([
        dbc.Col(chart_card("NAV over time", "Price of one unit. Direct sits slightly above Regular.",
                           dcc.Graph(id="nav-graph", figure=style_fig(fig_nav0, 340), config={"displayModeBar": False})), width=6),
        dbc.Col(chart_card("TER over time", "Yearly fee. The shaded gap is what the distributor's commission costs.",
                           dcc.Graph(id="ter-graph", figure=style_fig(fig_ter0, 340), config={"displayModeBar": False})), width=6),
    ], className="mb-4"),

    dbc.Row(dbc.Col(chart_card("The numbers behind the charts", "Direct vs Regular for this COMPANY", html.Div(table0, id="COMPANY-table")), width=12),
            className="mb-5"),
], fluid=True, className="px-4")

# one input -> five outputs (his "Pattern 3")
@app.callback(Output("COMPANY-title", "children"), Output("COMPANY-cards", "children"),
              Output("nav-graph", "figure"), Output("ter-graph", "figure"), Output("COMPANY-table", "children"),
              Input("COMPANY-select", "value"))
def update_COMPANY(value):
    title, cards, f_nav, f_ter, table = profile(int(value))
    return title, cards, style_fig(f_nav, 340), style_fig(f_ter, 340), table

app.run(jupyter_mode="inline", jupyter_height=2300, port=8060, debug=True, dev_tools_hot_reload=False)

In [118]:
def tint(hex_color, amount, base=None):
    """Mix a colour into the card colour and return a solid hex code. amount 0 = card colour, 1 = full colour."""
    base = (base or CARDS).lstrip("#")
    c = hex_color.lstrip("#")
    mix = [round(int(base[i:i + 2], 16) * (1 - amount) + int(c[i:i + 2], 16) * amount) for i in (0, 2, 4)]
    return "#{:02X}{:02X}{:02X}".format(*mix)

In [ ]:
# ---------- Dashboard 2: what does the commission cost? ----------
GAP = "#F472B6"                              # pink: only for "the gap = commission"
SLIDER_TXT = "#8B98A5"

t2 = ter.dropna(subset=["R_TER", "D_TER"]).copy()
t2["gap"] = t2["R_TER"] - t2["D_TER"]
t2 = t2.sort_values("Date")

# ---------- helpers ----------
def marks(vals, f=str):
    """Slider marks in readable grey (the default dark text vanishes on a dark page)."""
    return {int(v): {"label": f(v), "style": {"color": SLIDER_TXT}} for v in vals}

def lift_zero(fig, ymax):
    """Clean y ticks starting at 0, with the axis floor slightly below 0 so '0' never touches the x labels."""
    mag = 10 ** np.floor(np.log10(ymax / 5))
    step = next(m * mag for m in (1, 2, 5, 10) if m * mag >= ymax / 5)
    ticks = np.round(np.arange(0, ymax + step * 0.01, step), 6)
    fig.update_yaxes(range=[-ymax * 0.07, ymax], tickmode="array", tickvals=ticks.tolist())
    fig.update_xaxes(automargin=True)
    fig.update_yaxes(automargin=True)

def inr(lakh):
    return f"₹{lakh / 100:.2f} crore" if lakh >= 100 else f"₹{lakh:.1f} lakh"

# ---------- KPIs: each COMPANY's latest gap ----------
now2 = (t2.groupby("MF_ID").tail(1)
          .merge(t2.groupby("MF_ID").head(1)[["MF_ID", "gap"]].rename(columns={"gap": "gap_first"}), on="MF_ID")
          .merge(companies[["MF_ID", "COMPANY"]], on="MF_ID"))
avg_gap2  = now2["gap"].mean()
share     = avg_gap2 / now2["R_TER"].mean() * 100
widest    = now2.loc[now2["gap"].idxmax()]
narrowest = now2.loc[now2["gap"].idxmin()]
n_narrowed, n_total = int((now2["gap"] < now2["gap_first"]).sum()), len(now2)

# ---------- chart A: gap over time, all companies together ----------
m = (t2.assign(month=t2["Date"].dt.to_period("M").dt.to_timestamp())
       .groupby(["MF_ID", "month"])["gap"].mean().reset_index())
band = (m.groupby("month")["gap"]
          .agg(med="median", q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75), n="count")
          .reset_index())
band = band[band["n"] >= 5]

fig_gap = go.Figure()
fig_gap.add_trace(go.Scatter(x=band["month"], y=band["q25"], mode="lines", line=dict(width=0),
                             hoverinfo="skip", showlegend=False))
fig_gap.add_trace(go.Scatter(x=band["month"], y=band["q75"], mode="lines", line=dict(width=0),
                             fill="tonexty", fillcolor="tint(244,114,182,0.20)",
                             hoverinfo="skip", showlegend=False))
fig_gap.add_trace(go.Scatter(x=band["month"], y=band["med"], mode="lines",
                             line=dict(color=GAP, width=3, shape="spline", smoothing=0.6),
                             customdata=band["n"],
                             hovertemplate="<b>%{y:.2f} pp</b> median gap · %{customdata} companies<extra></extra>"))
lastb = band.iloc[-1]
fig_gap.add_trace(go.Scatter(x=[lastb["month"]], y=[lastb["med"]], mode="markers+text",
                             text=[f"  {lastb['med']:.2f} pp"], textposition="middle right",
                             textfont=dict(color=GAP, size=14), marker=dict(color=GAP, size=9),
                             cliponaxis=False, hoverinfo="skip", showlegend=False))
fig_gap.update_layout(showlegend=False, margin=dict(r=90))
fig_gap.update_xaxes(range=[band["month"].iloc[0], lastb["month"]])
fig_gap.update_yaxes(ticksuffix=" pp")
lift_zero(fig_gap, band["q75"].max() * 1.12)

# ---------- chart B: gap by fund COMPANY, for a chosen year ----------
yr_gap = (t2.assign(year=t2["Date"].dt.year)
            .groupby(["MF_ID", "year"]).agg(gap=("gap", "mean"), D_TER=("D_TER", "mean"), R_TER=("R_TER", "mean"))
            .reset_index().merge(companies[["MF_ID", "COMPANY"]], on="MF_ID"))
y0, y1 = int(yr_gap["year"].min()), int(yr_gap["year"].max())
gmax = yr_gap["gap"].max()
rank_height = max(420, 26 * int(yr_gap.groupby("year")["MF_ID"].nunique().max()))
BAR_SCALE = [[0,"#FBCFE8" ], [0.5, "#EC5AA3"], [1, "#E6397E"]]   # dark rose -> pink -> light blush

def rank_title(year):
    return f"Gap by fund COMPANY, {year}" + (" so far" if year == y1 else "")

def rank_fig(year):
    d = yr_gap[yr_gap["year"] == year].sort_values("gap")               # biggest ends up on top
    fig = go.Figure(go.Bar(
        x=d["gap"], y=d["COMPANY"], orientation="h",
        marker=dict(color=d["gap"], colorscale=BAR_SCALE, cmin=0, cmax=gmax, showscale=False),
        text=d["gap"].map("{:.2f}".format), textposition="outside", cliponaxis=False,
        customdata=np.stack([d["D_TER"], d["R_TER"]], axis=-1),
        hovertemplate="<b>%{y}</b><br>Gap %{x:.2f} pp<br>Direct %{customdata[0]:.2f}% · Regular %{customdata[1]:.2f}%<extra></extra>"))
    fig.update_layout(margin=dict(r=60), bargap=0.3, showlegend=False)
    fig.update_xaxes(ticksuffix=" pp", range=[0, gmax * 1.18])           # same range every year, so years compare
    fig.update_yaxes(showgrid=False, tickfont=dict(size=11), automargin=True)
    return fig

def calc(amount, years, gross):
    yrs = np.arange(0, years + 1)
    d_val = amount * (1 + (gross - avg_d) / 100) ** yrs               # in ₹ lakh
    r_val = amount * (1 + (gross - avg_r) / 100) ** yrs
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=yrs, y=d_val, name="Direct", mode="lines",
                             line=dict(color=DIRECT, width=3), hovertemplate="%{y:.2f} lakh"))
    fig.add_trace(go.Scatter(x=yrs, y=r_val, name="Regular", mode="lines",
                             line=dict(color=REGULAR, width=3),
                             fill="tonexty", fillcolor=tint(GAP, 0.38),       # brighter band = cost of commission
                             hovertemplate="%{y:.2f} lakh"))
    for name, col, val in [("Direct", DIRECT, d_val[-1]), ("Regular", REGULAR, r_val[-1])]:
        fig.add_trace(go.Scatter(x=[years], y=[val], mode="markers+text", text=[f"  {name} {val:.1f}"],
                                 textposition="middle right", textfont=dict(color=col, size=13),
                                 marker=dict(color=col, size=9), cliponaxis=False,
                                 hoverinfo="skip", showlegend=False))
    mid = (d_val[-1] + r_val[-1]) / 2
    fig.add_annotation(x=years, y=mid, text=f"  lost {inr(d_val[-1] - r_val[-1])}", showarrow=False,
                       xanchor="left", font=dict(color=GAP, size=12))
    fig.update_layout(hovermode="x unified", showlegend=False, margin=dict(r=150))
    fig.update_xaxes(title_text="Years invested", title_standoff=10, range=[0, years])
    fig.update_yaxes(title_text="Value (₹ lakh)", title_standoff=10)
    lift_zero(fig, d_val[-1] * 1.12)
    lost = d_val[-1] - r_val[-1]
    text = (f"On **{inr(amount)}** at **{gross}%** a year before costs, after **{years} years** Direct is worth "
            f"**{inr(d_val[-1])}** and Regular **{inr(r_val[-1])}**. "
            f"The commission costs about **{inr(lost)}** ({lost / d_val[-1] * 100:.0f}% of the Direct value).")
    return fig, text

ValueError: 
    Invalid value of type 'builtins.str' received for the 'fillcolor' property of scatter
        Received value: 'tint(244,114,182,0.20)'

    The 'fillcolor' property is a color and may be specified as a string in the following formats:
      - hex or short hex (e.g. '#d3d3d3', '#d3d')
      - hex or short hex with alpha (e.g. '#d3d3d380', '#d3d8')
      - rgb (e.g. 'rgb(255, 0, 0)', 'rgb(255 0 0)')
      - rgba (e.g. 'rgba(255, 0, 0, 0.5)', 'rgba(255 0 0 / 0.5)')
      - hsl (e.g. 'hsl(0, 100%, 50%)', 'hsl(0deg 100% 50%)')
      - hsla (e.g. 'hsla(0, 100%, 50%, 0.5)', 'hsla(0deg 100% 50% / 0.5)')
      - hwb (e.g. 'hwb(0 0% 100%)')
      - lab/lch/oklab/oklch (e.g. 'oklch(0.7 0.15 180)')
      - color (e.g. 'color(display-p3 1 0 0)')
      - named colors (full list: https://www.w3.org/TR/css-color-4/#named-color)
      - Any other supported CSS 4 color format: https://www.w3.org/TR/css-color-4/

name 'rgba' is not defined


In [ ]:
app2 = Dash(__name__, external_stylesheets=[THEME])
fig_rank0 = style_fig(rank_fig(y1), rank_height)
dbc.Col(kpi("Widest gap", f"{widest['gap']:.2f} pp", sub=widest["COMPANY"]), width=3),
dbc.Col(kpi("Narrowest gap", f"{narrowest['gap']:.2f} pp", sub=narrowest["COMPANY"]), width=3),

app2.layout = dbc.Container([
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 2 · What does the distributor's commission cost?", className="text-muted")],
                    width=12), className="mb-3"),

    dbc.Row([
        dbc.Col(kpi("Average gap today", f"{avg_gap2:.2f} pp", GAP, "Regular TER minus Direct TER"), width=3),
        dbc.Col(kpi("Commission share", f"{share:.0f}%", GAP, "of the Regular TER is the gap"), width=3),
        dbc.Col(kpi("Widest gap", f"{widest['gap']:.2f} pp", sub=widest["COMPANY"]), width=3),
        dbc.Col(kpi("Narrowest gap", f"{narrowest['gap']:.2f} pp", sub=narrowest["COMPANY"]), width=3),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(chart_card("How the gap has moved over time",
                           "Line = middle fund COMPANY. Shaded band = the middle half of fund COMPANYs.",
                           graph(fig_gap, 340)), width=8),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("What this tells us"),
            dcc.Markdown(f"""
- On average, **{avg_gap2:.2f} pp** of the Regular fee goes to the distributor, about **{share:.0f}%** of it.
- The gap **narrowed in {n_narrowed} of {n_total}** fund COMPANYs since their first reading.
- **{widest['COMPANY']}** has the widest gap ({widest['gap']:.2f} pp), **{narrowest['COMPANY']}** the narrowest ({narrowest['gap']:.2f} pp).
- A small yearly gap still adds up, because it compounds. See the calculator below.
            """)]), className="h-100"), width=4),
    ], className="mb-4"),

    dbc.Row([
        # ---- left: ranking with a year slider ----
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5(rank_title(y1), id="rank-title", className="mb-0"),
            html.Small("Average Regular minus Direct TER in that year. Drag the slider to change the year.",
                       className="text-muted"),
            html.Div(dcc.Slider(id="rank-year", min=y0, max=y1, step=1, value=y1, marks=marks(range(y0, y1 + 1))),
                     className="mt-3 mb-2"),
            dcc.Graph(id="rank-graph", figure=fig_rank0, config={"displayModeBar": False}),
        ]), className="h-100"), width=6),

        # ---- right: cost calculator with three sliders ----
        dbc.Col(chart_card("What the gap costs over the years",
                           "Uses today's average Direct and Regular TER across all fund COMPANYs.", html.Div([
            html.Label("Amount invested (₹ lakh)", className="text-muted mt-2"),
            dcc.Slider(id="amount", min=1, max=100, step=1, value=10,
                       marks=marks([1, 10, 25, 50, 75, 100])),
            html.Label("Years invested", className="text-muted mt-3"),
            dcc.Slider(id="years", min=1, max=30, step=1, value=15,
                       marks=marks([1, 5, 10, 15, 20, 25, 30])),
            html.Label("Return before costs (% a year)", className="text-muted mt-3"),
            dcc.Slider(id="gross", min=6, max=18, step=1, value=12,
                       marks=marks([6, 9, 12, 15, 18])),
            dcc.Graph(id="calc-graph", figure=style_fig(fig_calc0, 340), config={"displayModeBar": False}),
            dcc.Markdown(text_calc0, id="calc-text"),
        ])), width=6),
    ], className="mb-5"),
], fluid=True, className="px-4")

@app2.callback(Output("rank-graph", "figure"), Output("rank-title", "children"), Input("rank-year", "value"))
def update_rank(year):
    return style_fig(rank_fig(year), rank_height), rank_title(year)

@app2.callback(Output("calc-graph", "figure"), Output("calc-text", "children"),
               Input("amount", "value"), Input("years", "value"), Input("gross", "value"))
def update_calc(amount, years, gross):
    fig, text = calc(amount, years, gross)
    return style_fig(fig, 340), text

app2.run(jupyter_mode="inline", jupyter_height=2100, port=8062, debug=True, dev_tools_hot_reload=False)

name 'rgba' is not defined


- DASHBOARD 3 

In [ ]:
# ---------- Dashboard 3: do Direct plans really earn more? ----------
# one row per company and date, with both plans' NAV side by side (D = Direct, R = Regular)
pv = (nav.dropna(subset=["NAV"])
        .pivot_table(index=["MF_ID", "Date"], columns="Plan", values="NAV")
        .rename(columns={"Direct Plan": "D", "Regular Plan": "R"})
        .dropna(subset=["D", "R"]).reset_index().sort_values(["MF_ID", "Date"]))
name_of = companies.set_index("MF_ID")["COMPANY"].to_dict()
END = pv["Date"].max()
WINDOWS = {"1 year": 1, "3 years": 3, "5 years": 5, "Since start": None}

def window_stats(years):
    """Annualised return of each plan over the last `years` years (or the full history if None)."""
    rows = []
    for mf, g in pv.groupby("MF_ID"):
        end = g["Date"].iloc[-1]
        if end < END - pd.Timedelta(days=30):            # stopped reporting, skip
            continue
        if years is not None:
            target = end - pd.DateOffset(years=years)
            g = g[g["Date"] >= target]
            if (g["Date"].iloc[0] - target).days > 10:   # not enough history for this window
                continue
        yrs = (end - g["Date"].iloc[0]).days / 365.25
        if yrs < 0.9:
            continue
        d_c = ((g["D"].iloc[-1] / g["D"].iloc[0]) ** (1 / yrs) - 1) * 100
        r_c = ((g["R"].iloc[-1] / g["R"].iloc[0]) ** (1 / yrs) - 1) * 100
        fee = t2[(t2["MF_ID"] == mf) & (t2["Date"] >= g["Date"].iloc[0])]["gap"].mean()
        rows.append(dict(MF_ID=mf, COMPANY=name_of.get(mf, str(mf)), years=yrs,
                         d=d_c, r=r_c, ret_gap=d_c - r_c, ter_gap=fee))
    return pd.DataFrame(rows, columns=["MF_ID", "COMPANY", "years", "d", "r", "ret_gap", "ter_gap"])

STATS = {k: window_stats(v) for k, v in WINDOWS.items()}

# ---------- KPI row ----------
def kpi_row(st):
    n, ahead = len(st), int((st["ret_gap"] > 0).sum())
    return dbc.Row([
        dbc.Col(kpi("Median Direct return", f"{st['d'].median():.1f}%", DIRECT, "a year, annualised"), width=3),
        dbc.Col(kpi("Median Regular return", f"{st['r'].median():.1f}%", REGULAR, "a year, annualised"), width=3),
        dbc.Col(kpi("Median return gap", f"{st['ret_gap'].median():.2f} pp", GAP, "Direct minus Regular, per year"), width=3),
        dbc.Col(kpi("Direct ahead in", f"{ahead} of {n}", sub="fund companies in this window"), width=3),
    ], className="mb-4")

# ---------- chart A: dumbbell, Regular vs Direct return for each company ----------
def dumb_fig(st):
    d = st.sort_values("d")                                # best Direct return ends up on top
    ys = d["COMPANY"].tolist()
    lx, ly = [], []
    for c, a, b in zip(ys, d["r"], d["d"]):                # one pink connector per company
        lx += [a, b, None]; ly += [c, c, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lx, y=ly, mode="lines", line=dict(color=rgba(GAP, 0.6), width=4),
                             hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=d["r"], y=ys, mode="markers", name="Regular",
                             marker=dict(color=REGULAR, size=11),
                             hovertemplate="<b>%{y}</b><br>Regular %{x:.1f}% a year<extra></extra>"))
    fig.add_trace(go.Scatter(x=d["d"], y=ys, mode="markers", name="Direct",
                             marker=dict(color=DIRECT, size=11),
                             hovertemplate="<b>%{y}</b><br>Direct %{x:.1f}% a year<extra></extra>"))
    fig = style_fig(fig, max(420, 26 * len(d) + 80))
    fig.update_layout(margin=dict(t=40), legend=dict(orientation="h", x=0, y=1.0, yanchor="bottom"))
    fig.update_xaxes(showgrid=True, gridcolor=GRID, ticksuffix="%", title_text="Annualised return (% a year)")
    fig.update_yaxes(categoryorder="array", categoryarray=ys, showgrid=False,
                     tickfont=dict(size=11), automargin=True)
    return fig

# ---------- chart B: does the fee gap show up in the returns? ----------
def scatter_fig(st):
    s = st.dropna(subset=["ter_gap"])
    lo = min(0, s["ter_gap"].min(), s["ret_gap"].min()) - 0.1
    hi = max(s["ter_gap"].max(), s["ret_gap"].max()) * 1.08
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                             line=dict(color=MUTED, width=1.5, dash="dot"), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=s["ter_gap"], y=s["ret_gap"], mode="markers", text=s["COMPANY"],
                             marker=dict(color=GAP, size=11, line=dict(color=CARDS, width=1)),
                             hovertemplate="<b>%{text}</b><br>Fee gap %{x:.2f} pp<br>Return gap %{y:.2f} pp<extra></extra>"))
    fig.add_annotation(x=hi, y=hi, text="return gap = fee gap", showarrow=False,
                       xanchor="right", yshift=-14, font=dict(color=MUTED, size=12))
    fig = style_fig(fig, 380)
    fig.update_layout(showlegend=False, margin=dict(r=30))
    fig.update_xaxes(range=[lo, hi], title_text="Fee gap: Regular TER minus Direct TER (pp a year)",
                     showgrid=True, gridcolor=GRID, title_standoff=10)
    fig.update_yaxes(range=[lo, hi], title_text="Return gap: Direct minus Regular (pp a year)", title_standoff=10)
    return fig

def window_text(st, label):
    n, ret, fee = len(st), st["ret_gap"].median(), st["ter_gap"].median()
    extra = ("\n- For **Since start**, each company is measured over its own full history, so the lengths differ."
             if label == "Since start" else "")
    return f"""
- **{n}** fund companies have enough history for **{label.lower()}**. The others are left out.
- Direct earned a median **{ret:.2f} pp** a year more than Regular.
- The fee gap (TER) over the same period had a median of **{fee:.2f} pp**.
- When the dots sit close to the dotted line, the fee gap explains the return gap.{extra}
"""

# ---------- chart C: what ₹10 lakh really became, for one company ----------
def growth_fig(mf):
    g = pv[pv["MF_ID"] == mf]
    d_val, r_val = g["D"] / g["D"].iloc[0] * 10, g["R"] / g["R"].iloc[0] * 10        # in ₹ lakh
    last_x = g["Date"].iloc[-1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=g["Date"], y=d_val, name="Direct", mode="lines",
                             line=dict(color=DIRECT, width=3), hovertemplate="%{y:.2f} lakh"))
    fig.add_trace(go.Scatter(x=g["Date"], y=r_val, name="Regular", mode="lines",
                             line=dict(color=REGULAR, width=3),
                             fill="tonexty", fillcolor=rgba(GAP, 0.38), hovertemplate="%{y:.2f} lakh"))
    for name, col, val in [("Direct", DIRECT, d_val.iloc[-1]), ("Regular", REGULAR, r_val.iloc[-1])]:
        fig.add_trace(go.Scatter(x=[last_x], y=[val], mode="markers+text", text=[f"  {name} {val:.1f}"],
                                 textposition="middle right", textfont=dict(color=col, size=13),
                                 marker=dict(color=col, size=9), cliponaxis=False,
                                 hoverinfo="skip", showlegend=False))
    lost = d_val.iloc[-1] - r_val.iloc[-1]
    fig.add_annotation(x=last_x, y=(d_val.iloc[-1] + r_val.iloc[-1]) / 2, text=f"  behind by {inr(lost)}",
                       showarrow=False, xanchor="left", font=dict(color=GAP, size=12))
    fig.update_layout(hovermode="x unified", showlegend=False, margin=dict(r=170))
    fig.update_xaxes(range=[g["Date"].iloc[0], last_x])
    fig.update_yaxes(title_text="Value (₹ lakh)", title_standoff=10)
    text = (f"₹10 lakh put into **{name_of[mf]}** on **{g['Date'].iloc[0]:%d %b %Y}** would be worth "
            f"**{inr(d_val.iloc[-1])}** in Direct and **{inr(r_val.iloc[-1])}** in Regular by "
            f"**{last_x:%d %b %Y}**. Regular is behind by **{inr(lost)}**.")
    return fig, text

In [ ]:
app3 = Dash(__name__, external_stylesheets=[THEME])

co_ids = sorted(pv["MF_ID"].unique(), key=lambda m: name_of.get(m, "").lower())
co_options = [{"label": name_of.get(m, str(m)), "value": int(m)} for m in co_ids]
co_default = int(next((m for m in co_ids if name_of.get(m) == "HDFC"), co_ids[0]))

W0 = "3 years"
st0 = STATS[W0]
fig_g0, text_g0 = growth_fig(co_default)

app3.layout = dbc.Container([
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 3 · Do Direct plans really earn more?", className="text-muted")],
                    width=12), className="mb-3"),

    # ---- window picker ----
    dbc.Row([
        dbc.Col([html.H4("Pick a time window", className="mb-0"),
                 html.Small("Only companies with enough history for the window are compared.", className="text-muted")], md=6),
        dbc.Col(dbc.RadioItems(id="window", options=[{"label": k, "value": k} for k in WINDOWS], value=W0,
                               className="btn-group", inputClassName="btn-check",
                               labelClassName="btn btn-outline-light", labelCheckedClassName="active"),
                md=6, className="text-md-end"),
    ], className="mb-3 align-items-end"),

    html.Div(kpi_row(st0), id="kpis"),

    dbc.Row([
        dbc.Col(chart_card("Return of each fund company, Direct vs Regular",
                           "Blue = Direct, orange = Regular. The pink line is what Regular gives up each year.",
                           dcc.Graph(id="dumb-graph", figure=dumb_fig(st0), config={"displayModeBar": False})),
                width=7),
        dbc.Col([
            chart_card("Does the fee gap show up in returns?",
                       "Each dot is a fund company.",
                       dcc.Graph(id="scatter-graph", figure=scatter_fig(st0), config={"displayModeBar": False})),
            dbc.Card(dbc.CardBody([html.H5("What this tells us"),
                                   dcc.Markdown(window_text(st0, W0), id="ret-text")]), className="mt-3"),
        ], width=5),
    ], className="mb-4"),

    # ---- one company, real growth ----
    dbc.Row([
        dbc.Col([html.H4("What ₹10 lakh really became", className="mb-0"),
                 html.Small("Pick a fund company to see its full history, Direct vs Regular.", className="text-muted")], md=8),
        dbc.Col(dcc.Dropdown(id="co-select", options=co_options, value=co_default, clearable=False), md=4),
    ], className="mb-3 align-items-end"),
    dbc.Row(dbc.Col(chart_card("Growth of ₹10 lakh", "Same money, same day, two plans. The pink band is the cost of the commission.",
                               html.Div([dcc.Graph(id="growth-graph", figure=style_fig(fig_g0, 380),
                                                   config={"displayModeBar": False}),
                                         dcc.Markdown(text_g0, id="growth-text")])), width=12),
            className="mb-5"),
], fluid=True, className="px-4")

@app3.callback(Output("kpis", "children"), Output("dumb-graph", "figure"),
               Output("scatter-graph", "figure"), Output("ret-text", "children"),
               Input("window", "value"))
def update_window(label):
    st = STATS[label]
    return kpi_row(st), dumb_fig(st), scatter_fig(st), window_text(st, label)

@app3.callback(Output("growth-graph", "figure"), Output("growth-text", "children"), Input("co-select", "value"))
def update_growth(mf):
    fig, text = growth_fig(int(mf))
    return style_fig(fig, 380), text

app3.run(jupyter_mode="inline", jupyter_height=2300, port=8063, debug=True, dev_tools_hot_reload=False)

NameError: name 'rgba' is not defined